In [2]:
import os
import json
import pandas as pd

folder_path = "/kaggle/input/cmdataset-1m/ctssb_data_1M/ctssb_data_1M/"

data = []

# Loop through all files in the folder
for filename in os.listdir(folder_path):
    if filename.lower().endswith((".json", ".jsonl")):  # Only JSON/JSONL files
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:  # Works for both .jsonl (multiple lines) and .json (single line)
                data.append(json.loads(line))

print(f"Total variants from all JSON/JSONL files: {len(data)}")

# Optional: convert to DataFrame for analysis
df = pd.DataFrame(data)
print(df.head())


Total variants from all JSON/JSONL files: 966985
    project                                commit_sha  \
0  pVAC-Seq  81d85c18a3a309b9a69a4066c32ce954f4090f3a   
1  pVAC-Seq  5639a41cea031bda39970f09bb7724d08cf43726   
2  pVAC-Seq  67ee103206f59e8cfa8364c284c3c1aa30541ae5   
3  pVAC-Seq  e5d9317fdda7538b3fae64bbb629ed51110ae5f8   
4  pVAC-Seq  cd086269eb1ccd2281382f4e1dea6427a1aae847   

                                 parent_sha  \
0  d6342af96f5bc274beb8b367390dfdc3a7e68465   
1  85cce57d4c0702639d0499fe9a8cbe5da31aa22c   
2  0e1640127f50658ead2d9ed50b0b4c231f8b94a0   
3  030556dea3be9ebbd59a1b5bd9d371c789de2e7c   
4  ed789c12b78e07bfc34ee373c0ae8120b12f1968   

                               file_path                         project_url  \
0                    pvacseq/lib/main.py  https://github.com/fw1121/pVAC-Seq   
1                pvacseq/lib/pipeline.py  https://github.com/fw1121/pVAC-Seq   
2                pvacseq/lib/pipeline.py  https://github.com/fw1121/pVAC-Seq   
3    

In [3]:
# Keep only the necessary columns for our seq2seq task
df_clean = df[['before', 'after']].copy()

# Drop rows where 'before' or 'after' columns are empty or null
df_clean = df_clean.dropna()
df_clean = df_clean[(df_clean['before'].str.strip() != "") & (df_clean['after'].str.strip() != "")]

# Normalize whitespace (remove leading/trailing spaces)
df_clean['before'] = df_clean['before'].str.strip()
df_clean['after'] = df_clean['after'].str.strip()

# Shuffle the dataset to ensure the model doesn't learn any order-based patterns
df_clean = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)

# Preview the cleaned data
print(f"Total examples after cleaning: {len(df_clean)}")
print("\nCleaned Data Head:")
print(df_clean.head(3))

Total examples after cleaning: 966985

Cleaned Data Head:
                                  before  \
0       print ( dumpdata ( bot . env ) )   
1  unique_together = [ 'code' , 'name' ]   
2             sqlite_default = ':memory'   

                                               after  
0                       print ( dump ( bot . env ) )  
1  unique_together = [ 'code' , 'name' , 'categor...  
2                        sqlite_default = ':memory:'  


In [4]:
from sklearn.model_selection import train_test_split

# First, split into 80% train and 20% temporary
train_df, temp_df = train_test_split(df_clean, test_size=0.2, random_state=42)

# Then, split the temporary set into 50% validation and 50% test (10% of original each)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train examples: {len(train_df)}")
print(f"Validation examples: {len(val_df)}")
print(f"Test examples: {len(test_df)}")

Train examples: 773588
Validation examples: 96698
Test examples: 96699


In [5]:
from datasets import Dataset

train_dataset_hf = Dataset.from_pandas(train_df[['before','after']])
val_dataset_hf   = Dataset.from_pandas(val_df[['before','after']])
test_dataset_hf  = Dataset.from_pandas(test_df[['before','after']])

In [ ]:
# !pip install datasets
# 

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Salesforce/codet5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [7]:
max_length = 128

def tokenize_function(batch):
    """Tokenizes the 'before' and 'after' text in a batch."""
    return tokenizer(
        batch['before'],
        text_target=batch['after'],
        padding='max_length',
        truncation=True,
        max_length=max_length
    )

# Apply the tokenization to all datasets
train_dataset_tokenized = train_dataset_hf.map(tokenize_function, batched=True)
val_dataset_tokenized   = val_dataset_hf.map(tokenize_function, batched=True)
test_dataset_tokenized  = test_dataset_hf.map(tokenize_function, batched=True)

# Set the format to PyTorch tensors for the Trainer
train_dataset_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/773588 [00:00<?, ? examples/s]

Map:   0%|          | 0/96698 [00:00<?, ? examples/s]

Map:   0%|          | 0/96699 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./codet5-research",
    eval_strategy="steps",
    eval_steps=5000,                 # evaluate periodically
    logging_steps=1000,              # log metrics every 1000 steps
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,   # simulates batch size 32 without extra memory
    save_steps=10000,                # save checkpoints less frequently
    save_total_limit=3,              
    learning_rate=5e-5,
    num_train_epochs=2,              # start with 2 epochs, can increase if eval_loss decreases
    weight_decay=0.01,
    fp16=True,                        # mixed precision for faster training
    report_to="none",
    load_best_model_at_end=True,     # loads checkpoint with best eval metric
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
    tokenizer=tokenizer
)

trainer.train()


In [ ]:
print("--- Evaluating on Test Set ---")
results = trainer.evaluate(test_dataset_tokenized)
print(results)

In [ ]:
trainer.save_model("my_debugger_model")
tokenizer.save_pretrained("my_debugger_model")


# from transformers import Trainer, TrainingArguments

# training_args = TrainingArguments(
#     output_dir="./codet5-debugger",
#     eval_strategy="steps",
#     eval_steps=2000,
#     logging_steps=500,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     gradient_accumulation_steps = 1, 
#     save_steps=5000,
#     save_total_limit=2,
#     learning_rate=5e-5,
#     num_train_epochs=1,
#     weight_decay=0.01,
#     fp16=True,
#     report_to="none",
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset_tokenized,
#     eval_dataset=val_dataset_tokenized,
#     tokenizer=tokenizer
# )

# # Start the training process
# trainer.train()